# GitHub Triager RL Training (GRPO)

This notebook trains a Llama 3.2 model to triage GitHub issues using Reinforcement Learning (GRPO).

**Prerequisites:**
1. Ensure your Hugging Face Space is running the updated server code.
2. Set your environment URL in the configuration cell below.

In [ ]:
# 1. Install Dependencies
!pip install unsloth vllm
!pip install --no-deps trl peft accelerate diffusers
!pip install httpx

In [ ]:
import os
import re
import json
import time
import requests
import torch
import random
import matplotlib.pyplot as plt
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

# --- DIRECTORY SETUP ---
os.makedirs("results", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# --- CONFIGURATION ---
ENVIRONMENT_URL = "https://kavya011-github-triager-rl.hf.space"
MODEL_NAME      = "unsloth/Llama-3.2-3B-Instruct"
MAX_STEPS       = 250      
BATCH_SIZE      = 1        
NUM_GENERATIONS = 4        
GRADIENT_ACCUMULATION_STEPS = 4
MAX_SEQ_LENGTH = 1024

## 1. Utility Functions

In [ ]:
def get_env_reward(completion: str) -> float:
    """Mirroring train_github_triager.py reward logic"""
    try:
        match = re.search(r'\{.*\}', str(completion), re.DOTALL)
        if not match: return 0.01
        
        action_data = json.loads(match.group(0))
        url = f"{ENVIRONMENT_URL}/grade/label_classification"
        response = requests.post(url, json={"action": action_data}, timeout=10)
        
        if response.status_code == 200:
            return float(response.json().get("score", 0.01))
    except: pass
    return 0.01

def compute_reward(prompts, completions, **kwargs):
    return [get_env_reward(c[0]["content"] if isinstance(c, list) else c) for c in completions]

def test_connection():
    print(f"Testing connection to {ENVIRONMENT_URL}...")
    try:
        resp = requests.get(f"{ENVIRONMENT_URL}/health")
        if resp.status_code == 200:
            print("✅ Successfully connected to the environment!")
            return True
    except Exception as e:
        print(f"❌ Connection failed: {e}")
    return False

## 2. Dataset and Model Setup

In [ ]:
def build_dataset(tokenizer, n_samples: int = 100):
    print(f"Collecting {n_samples} samples...")
    samples = []
    for i in range(n_samples):
        try:
            resp = requests.post(f"{ENVIRONMENT_URL}/reset?task_id=label_classification")
            obs = resp.json()["observation"]
            
            # Prompt logic identical to train_github_triager.py
            prompt_text = f"You are a GitHub issue triager.\nIssue Title: {obs['issue']['title']}\nIssue Body: {obs['issue']['body']}\nClassify this issue. Respond with JSON only.\nFormat: {{\"label\": \"<bug|feature|documentation|question|enhancement>\"}}"
            
            messages = [{"role": "user", "content": prompt_text}]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            samples.append({"prompt": prompt})
            if i % 10 == 0: print(f"  Progress: {i}/{n_samples}")
        except: time.sleep(1)
    return Dataset.from_list(samples)

def evaluate_model(model, tokenizer, dataset, n_episodes=20):
    print(f"Evaluating over {n_episodes} episodes...")
    total = 0.0
    for _ in range(n_episodes):
        prompt = dataset[random.randint(0, len(dataset)-1)]["prompt"]
        inputs  = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.1)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):]
        total += get_env_reward(response)
    return total / n_episodes

In [ ]:
# Initialize Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    use_gradient_checkpointing="unsloth",
)

test_connection()
dataset = build_dataset(tokenizer, 100)

## 3. Baseline Evaluation

In [ ]:
print("Running baseline evaluation...")
baseline_avg = evaluate_model(model, tokenizer, dataset, 20)
print(f"Baseline Average Reward: {baseline_avg:.4f}")

## 4. GRPO Training

In [ ]:
training_args = GRPOConfig(
    output_dir="./outputs/github-triager-grpo",
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_steps=MAX_STEPS,
    learning_rate=1e-5,
    logging_steps=5,
    save_steps=50,
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[compute_reward],
    tokenizer=tokenizer,
)

trainer.train()

## 5. Results and Visualization

In [ ]:
log_history = trainer.state.log_history
steps = [x["step"] for x in log_history if "loss" in x]
losses = [x["loss" ] for x in log_history if "loss" in x]
r_steps = [x["step"] for x in log_history if "reward" in x]
rewards = [x["reward"] for x in log_history if "reward" in x]

plt.figure(figsize=(15, 5))

# 1. Training Loss
plt.subplot(1, 3, 1)
plt.plot(steps, losses, color="royalblue")
plt.title("Training Loss")
plt.xlabel("Step")
plt.grid(True, alpha=0.3)

# 2. Reward Progress
plt.subplot(1, 3, 2)
plt.plot(r_steps, rewards, color="seagreen")
plt.title("Average Reward")
plt.xlabel("Step")
plt.grid(True, alpha=0.3)

# 3. Before vs After Comparison
print("Running post-training evaluation...")
trained_avg = evaluate_model(model, tokenizer, dataset, 20)

plt.subplot(1, 3, 3)
plt.bar(["Baseline", "Trained"], [baseline_avg, trained_avg], color=["#ff6b6b", "#51cf66"], edgecolor="black")
plt.ylabel("Average Reward")
plt.title("Comparison")

plt.tight_layout()
plt.savefig("results/training_results.png", dpi=150)
plt.show()

model.save_pretrained("outputs/github-triager-final")
tokenizer.save_pretrained("outputs/github-triager-final")
print("✅ All results and models saved in results/ and outputs/ directories.")